In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

In [2]:
X_user = pd.read_csv("../data/X_user.csv")
X_movie = pd.read_csv("../data/X_movie.csv")
y = pd.read_csv("../data/y.csv")

In [3]:
#split into train and test sets

In [4]:
#scale training data
X_user_unscaled = X_user
X_movie_unscaled = X_movie
y_unscaled = y

#data leakage? fit on training data only, then transform test data
scaler_users = StandardScaler()
scaler_users.fit(X_user_unscaled)
X_user = scaler_users.transform(X_user_unscaled)

scaler_movies = StandardScaler()
scaler_movies.fit(X_movie_unscaled)
X_movie = scaler_movies.transform(X_movie_unscaled)

scaler_y = MinMaxScaler((-1, 1))
y_unscaled = y_unscaled.values.reshape(-1, 1) # Reshape to 2D array for scaler to pass into fit and transfomr
scaler_y.fit(y_unscaled)
y = scaler_y.transform(y_unscaled)

#check inverse transform 
# user matix has NaN values
# first edition set NaN values to 0
# improvement to test - set existence array with 1 or 0 to show existence (improves cold start but pot not useful for ranking)
print(np.allclose(X_user_unscaled, scaler_users.inverse_transform(X_user), equal_nan=True))
print(np.allclose(X_movie_unscaled, scaler_movies.inverse_transform(X_movie)))
print(np.allclose(y_unscaled, scaler_y.inverse_transform(y)))


True
True
True


In [5]:
#set NaN values to 0
X_user[np.isnan(X_user)] = 0

In [6]:
print("X_user[0]:", X_user[0], "\nX_movie[0]:", X_movie[0], "\ny[0]:", y[0])


X_user[0]: [ 0.39210181  0.29376596  0.2848368  -1.25245578 -0.38574659  0.36571406
  0.          0.3064445  -0.35806297  0.63354306  0.46107947 -0.06346852
 -2.10269066  0.62253286 -0.19721082  0.65184086  0.62784483 -0.07758051
 -0.53863489] 
X_movie[0]: [-0.67127368 -0.56738576 -0.26483292 -0.29881588 -0.73014924 -0.45382141
 -0.10964296 -0.89122409 -0.3611915  -0.09999431 -0.27673211 -0.23072918
 -0.18974351  3.32011263 -0.46220148  2.08014898  1.62968011 -0.2363347
 -0.1383685   0.76910331] 
y[0]: [1.]


In [7]:
#split (before data leakage fix)

split = 0.8

X_user_train, X_user_test = X_user[:int(len(X_user)*split)], X_user[int(len(X_user)*split):]
X_movie_train, X_movie_test = X_movie[:int(len(X_movie)*split)], X_movie[int(len(X_movie)*split):]
y_train, y_test = y[:int(len(y)*split)], y[int(len(y)*split):]

In [26]:
#Build initial NN

from tensorflow.keras.layers import Dense

num_outputs = 32

tf.random.set_seed(42)

userNN = tf.keras.models.Sequential([
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(num_outputs, activation='linear')
])

movieNN = tf.keras.models.Sequential([
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(num_outputs, activation='linear')
])

#create user input and point to base network
input_user = tf.keras.layers.Input(shape=(X_user_train.shape[1],))
vu = userNN(input_user)
#normalise to unit vector
vu = tf.keras.ops.normalize(vu, axis=1)


#create movie input and point to base network
input_movie = tf.keras.layers.Input(shape=(X_movie_train.shape[1],))
vm = movieNN(input_movie)
#normalise to unit vector
vm = tf.keras.ops.normalize(vm, axis=1)

#dot product layer
output = tf.keras.layers.Dot(axes=1)([vu, vm])

#model spec
model = tf.keras.Model(inputs=[input_user, input_movie], outputs=output)

model.summary()



Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_10      │ (None, 19)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_12      │ (None, 20)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_6        │ (None, 32)        │     42,144 │ input_layer_10[0… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential_7        │ (None, 32)        │     42,400 │ input_layer_12[0… │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_4         │ (None, 32)        │          0 │ sequential_6[0][… │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalize_5         │ (None, 32)        │          0 │ sequential_7[0][… │
│ (Normalize)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot_2 (Dot)         │ (None, 1)         │          0 │ normalize_4[0][0… │
│                     │                   │            │ normalize_5[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 84,544 (330.25 KB)

 Trainable params: 84,544 (330.25 KB)

 Non-trainable params: 0 (0.00 B)

In [27]:
tf.random.set_seed(42)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,loss=cost_fn)

In [24]:
tf.random.set_seed(42)
model.fit([X_user_train, X_movie_train], y_train, epochs=30)

Epoch 1/30


4662/4662 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.1562
Epoch 2/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.1491
Epoch 3/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - loss: 0.1462
Epoch 4/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.1438
Epoch 5/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.1408
Epoch 6/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.1394
Epoch 7/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.1378
Epoch 8/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.1367
Epoch 9/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.1360
Epoch 10/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.1354
Epoch 11/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.1349
Epoch 12/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.1344
Epoch 13/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - loss: 0.1340
Epoch 14/30
4662/4662 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - loss: 0.1336
Epoch 15/30
4662/4662 ━━━━━━━━━━━━━━━━━━

In [25]:
#MSE eval on test set
model.evaluate([X_user_test, X_movie_test], y_test)

#test loss close to training loss, so model is not overfitting

1166/1166 ━━━━━━━━━━━━━━━━━━━━ 1s 953us/step - loss: 0.1526


0.15258891880512238